# <p align=center> Split Data </p>

### <p align=center> Extract different Molecular Component from PDBs with associated EDMs</p> 

In [ ]:
import os
import sys
sys.path.insert( 0, os.path.abspath("../../.."))
from typing import Callable
from copy import deepcopy

import gemmi

from xaidar.data.molecModels import createPDB, savePDB, sele_res, get_pdb_stats

In [2]:
import os
import sys
sys.path.append( os.path.abspath( "../../../" ) )
import gemmi

from xaidar.data.molecModels import createPDB, sele_res
iePDBPath = ("../../../data/ev2a/fragalysis/concatDir/aligned_files/"\
        "A0152a/A0152a.pdb")
pdb = gemmi.read_pdb( str(iePDBPath))


In [30]:
def createPDB( molecObj: gemmi.Structure | None = None, 
              modelList: list[ gemmi.Model]  | None = None,
              chainList: list[ gemmi.Chain ] | None = None,
              residSpan: gemmi.ResidueSpan | None = None, 
              atomList: list[gemmi.Atom]| None = None   ):
    """
    Only add a list with several items to the last argument of the hierarchy.
    """
    
    if not molecObj: new_molecObj = gemmi.Structure()
    if not modelList: new_model = [ gemmi.Model(1) ]
    if not chainList: new_chain = [ gemmi.Chain( "A") ]
    if not residSpan:
        new_residSpan = []
        resid = gemmi.Residue()
        resid.name = "MOL"
        new_residSpan.append( resid )
    
    if atomList:
        for atom in atomList:
            new_residSpan[0].add_atom( atom )
        residSpan = new_residSpan
    if  residSpan:
        for resid in residSpan: new_chain[0].add_residue( resid )
        chainList = new_chain
    if chainList:
        for chain in chainList: new_model[0].add_chain( chain )
        modelList = new_model
    if modelList:
        for model in modelList: new_molecObj.add_model( model )
        molecObj = new_molecObj
    return molecObj

createPDB( residSpan = ligList )

<gemmi.Structure  with 1 model(s)>

In [3]:
from pathlib import Path

import gemmi

def savePDB( structure: gemmi.Structure, outPath: Path | str):
    structure.write_pdb( str(outPath) )
    return None

savePath = Path( "../../../data/ev2a/testLig.pdb")
savePDB( ligObj, savePath )

In [10]:
def clear_empty(pdb):
    for model_id, model in enumerate(pdb):
        for chain in model:
            delete_list = [ resi_id for resi_id, residue in enumerate(chain) 
                                                        if len(residue) == 0]
            delete_list = delete_list[::-1]
            for resi_id in delete_list: del pdb[ model_id ][chain.name][resi_id]
    for model_id, model in enumerate(pdb):
        delete_list = [chain.name for chain in model if len(chain) == 0]
        for chain_name in delete_list: del pdb[ model_id ][chain_name]
    for model_idx, model in enumerate(pdb):
        if len(model) == 0: del pdb[model_idx]

    return pdb

In [ ]:
def sele_pdb(pdb: gemmi.Structure, level: str, selection : Callable, *args,
                                                        ) -> gemmi.Structure:
    """
    Perform filtering on a PDB structure based on a specified level and selection.
    Args:
    - pdb (gemmi.Structure): The PDB structure to filter.
    - level (str): The level of filtering ('model', 'chain', 'residue', 'atom').
    - selection (function): A function that takes an element of the specified level
      and additional arguments, returning True if the element should be included.
      - *args: Additional arguments to pass to the selection function.
      Returns:
      - list: A list of elements that meet the filtering selection.
    """
    
    empty_pdb = deepcopy( pdb ) # Store Object of models
    while len(empty_pdb) > 0: del empty_pdb[0] # Remove everything except Structure level info
    new_pdb = empty_pdb # Create new Structure Object to Store selected elements
  # Looking at models
    if level == 'model': 
        sele_models = selection( pdb, *args) # -> list[ gemmi.Model ]
        for sele_model in sele_models:
            new_pdb.add_model( sele_model ) # Fill Structure with selected Model Objects
    else:
        for model_id, model in enumerate(pdb):
            empty_model = gemmi.Model(model_id + 1 ) # Add Model attribute .num
            new_pdb.add_model( empty_model ) # Create Model Object without chains (empty)
  # Looking a chains
            new_model = new_pdb[-1] # Call Last Empty Model Object to Store chains in
            if level == 'chain': 
                sele_chains = selection( model, *args) # -> list[ gemmi.Chain ]
                for sele_chain in sele_chains:
                    new_model.add_chain( sele_chain )
            else:
                for chain in model:
                    empty_chain = gemmi.Chain(chain.name)
                    new_model.add_chain( empty_chain ) # Create Chain Object without residues
  # Looking at residues 
                    new_chain = new_model[chain.name] # Call Emtpy Chain Object to Store residue Objects in 
                    if level == 'residue':
                        sele_residues = selection( chain, *args) # list[ gemmi.Residue ]
                        for sele_residue in sele_residues: 
                            new_chain.add_residue( sele_residue )
                    else:
                        for residue in chain:
                            empty_resi = deepcopy( residue )
                            while len( empty_resi) > 0 : del empty_resi[0] # Only keep residue level info, delete all atoms
                            new_chain.add_residue( empty_resi ) # Create Residue Object without atoms
  # Looking at atoms                          
                            new_residue = new_chain[-1] # Store Object of atoms
                            if level == 'atom':
                                sele_atoms = selection( residue, *args) # list[ gemmi.Atom ]
                                # if sele_atoms != []:
                                for sele_atom in sele_atoms:
                                    new_residue.add_atom( sele_atom )
                            else:
                                raise ValueError(("Invalid level specified. "
                            "Choose from 'model', 'chain', 'residue', or 'atom'."))
    # Remove empty Models, Chains, Residues from resulting Structure    
    new_pdb = clear_empty(new_pdb)

    return new_pdb

## Extract Ligand

In [ ]:
ligList = sele_res( pdb, {"resname": ["LIG"]})
print( "Ligand: {}".format( ligList ) )

ligObj = createPDB( residSpan = ligList )
print( ligObj )

In [ ]:
import gemmi
import pathlib
from pathlib import Path


def extractLigands( pdb_st: gemmi.Structure, saveDirPath: pathlib.Path, protName: str ):
    """
    Extracts ligands from a PDB structure and saves them as individual PDB files.
    Args:
    - pdb_st (gemmi.Structure): The PDB structure object containing the ligands.
    - saveDirPath (pathlib.Path): The directory path where the ligand PDB files
    - protName (str): The name of the protein, used to name the ligand PDB files.
    Returns:
    - None: The function saves the ligand PDB files to the specified directory.
    """

    saveDirPath.mkdir( parents=True, exist_ok=True)
    if len(pdb_st) == 1:
        model = pdb_st[0]
        for chain in model:
            if chain.get_ligands():
                print( f"Processing Chain: {chain.name }")
                resSpan =  chain.whole() 
                ligName = list( set( resSpan.extract_sequence() ) )
                if len(ligName) > 1:
                    print( f"Bad arrangement of ligands with more than one in a chain: {ligName}" )
                    for lig in ligName:
                        ligPDBName = f"{protName}_{lig}.pdb"

                        for residue in chain.whole():
                            print(residue.name)
                            # Continue code

                else:
                    print( f"Processing Ligand: {ligName[0]}")
                    ligPDBName = f"{protName}_{ligName[0]}.pdb"
                    saveFilePath = saveDirPath / ligPDBName
                    saveFilePath = saveFilePath.resolve().as_posix().__str__()
                    create_ligand_file( resSpan, ligPDBName, saveFilePath)


    elif len(pdb_st) == 0:
        print( "Error with Model")
    else:
        print("More than one model")

In [120]:
def sele_Lig( chain: gemmi.Chain ):
    return [ res for res in chain if res.name == "LIG" ]

lig = sele_pdb( pdb, "residue", sele_Lig)

get_pdb_stats( lig )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 1
	Unique List of Non-A.A.: {'LIG'}


## Extract Protein

In [14]:
chain = pdb[0]["A"]
set([ res.name for res in chain  if not gemmi.find_tabulated_residue(res.name)])

{'LIG'}

In [3]:
get_pdb_stats( pdb )

Number of models: 1
Number of chains in 1st Model: 5

Chain ID: A
	Number of Residues: 141
	Unique List of Non-A.A.: {'LIG'}
	Contains A.A.
Chain ID: B
	Number of Residues: 1
	Unique List of Non-A.A.: {'ZN'}
Chain ID: C
	Number of Residues: 8
	Unique List of Non-A.A.: {'HOH'}
Chain ID: D
	Number of Residues: 3
	Unique List of Non-A.A.: {'DMS'}
Chain ID: E
	Number of Residues: 1
	Unique List of Non-A.A.: {'SO4'}


In [ ]:
from typing import Callable
def sele_pdb(pdb: gemmi.Structure, level: str, selection : Callable, *args):
    """
    Perform filtering on a PDB structure based on a specified level and selection.
    Args:
    - pdb (gemmi.Structure): The PDB structure to filter.
    - level (str): The level of filtering ('model', 'chain', 'residue', 'atom').
    - selection (function): A function that takes an element of the specified level
      and additional arguments, returning True if the element should be included.
      - *args: Additional arguments to pass to the selection function.
      Returns:
      - list: A list of elements that meet the filtering selection.
    """
    new_pdb = gemmi.Structure()
    new_pdb.name = pdb.name
    for model_id, model in enumerate(pdb):
        if level == 'model': 
            mew_models = 
            # selected_elements.append( selection(model, *args) )
            new_pdb.add_model( selection(model, *args) )
        else:
            new_pdb.add_model( gemmi.Model() )
            for chain in model:
                if level == 'chain': 
                    # selected_elements.append( selection(model, *args) )
                    new_pdb[model_id].add_chain( selection(chain, *args) )
                else:
                    new_pdb[model_id].add_chain( gemmi.Chain(chain.name) )
                    for residue in chain:
                        if level == 'residue':
                            # selected_elements.append( selection(residue, *args) )
                            new_pdb[model_id][chain.name].add_residue( 
                                selection(residue, *args) )
                        else:
                            for atom in residue:
                                if level == 'atom':
                                    selected_elements.append( selection(atom, *args) )

                                else:
                                    raise ValueError(("Invalid level specified. "
                            "Choose from 'model', 'chain', 'residue', or 'atom'."))

    return selected_elements
    





In [114]:
def clear_empty(pdb):
    for model_idx, model in enumerate(pdb):
        if len(model) == 0: del pdb[model_idx]
    for model_id, model in enumerate(pdb):
        delete_list = [chain.name for chain in model if len(chain) == 0]
        for chain_name in delete_list: del pdb[ model_id ][chain_name]
    for model_id, model in enumerate(pdb):
        for chain in model:
            delete_list = [ resi_id for resi_id, residue in enumerate(chain) 
                                                        if len(residue) == 0]
            delete_list = delete_list[::-1]
            for resi_id in delete_list: del pdb[ model_id ][chain.name][resi_id]
    return pdb

In [71]:
from copy import deepcopy
new_pdb = gemmi.Structure()
new_pdb.name = "test"
print( new_pdb )
# Model
empty_model = gemmi.Model(1)
new_pdb.add_model( empty_model )
new_model = new_pdb[0]
print( new_pdb , new_pdb[0], new_model)
# Chain
empty_chain = gemmi.Chain("A") 
new_model.add_chain( empty_chain )
empty_chain = gemmi.Chain("B")
new_model.add_chain( empty_chain )
new_chain = new_model["A"]
print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_chain)
# Residue
empty_resi = deepcopy(pdb[0]["A"][0])
for _ in empty_resi: del empty_resi[0] # Only keep residue level info, delete all atoms
new_chain.add_residue( empty_resi )
new_residue = new_chain[-1]
print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_pdb[0]["A"][0], new_residue,
      len(new_chain), len(new_residue) )

# new_chain.add_residue( gemmi.Residue() )
# print( new_pdb, new_pdb[0], new_pdb[0]["A"], new_pdb[0]["A"][0], new_chain)
# # new_model[ "A"].add_residue( gemmi.Residue() )
# res = gemmi.Residue()
# res.name, res.seqid = pdb[0]["A"][0].name, pdb[0]["A"][0].seqid

# res = deepcopy(pdb[0]["A"][0])
# def clearatoms( residue: gemmi.Residue) -> gemmi.Residue:
#     """
#     Remove all atoms from a residue.
#     Args:
#     - residue (gemmi.Residue): The residue from which to remove atoms.
#     Returns:
#     - gemmi.Residue: The residue with all atoms removed.
#     """
#     for index in range( len(residue)):  # Create a list to avoid modifying the collection while iterating
#         del residue[0]
#     return residue
# print( len(res) )
# res = clearatoms( res )
# print( len(res) , res.name, res.seqid  )

# # res.seqid = gemmi.SeqId(1, "d")
# # print( res.remove_atom() )
# print( pdb[0]["A"][-1])

<gemmi.Structure test with 0 model(s)>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 0 chain(s)> <gemmi.Model 1 with 0 chain(s)>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 2 chain(s)> <gemmi.Chain A with 0 res> <gemmi.Chain A with 0 res>
<gemmi.Structure test with 1 model(s)> <gemmi.Model 1 with 2 chain(s)> <gemmi.Chain A with 1 res> 7(SER) 7(SER) 1 0


In [121]:

def sele_AA( chain: gemmi.Chain ):
    return [ res for res in chain if gemmi.find_tabulated_residue(res.name).is_amino_acid() ]

prot = sele_pdb( pdb, "residue", sele_AA)

get_pdb_stats( prot )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


## Extract Water

In [118]:
def sele_HOH( chain: gemmi.Chain ):
    return [ res for res in chain if res.name == "HOH" ]

hoh = sele_pdb( pdb, "residue", sele_HOH)
get_pdb_stats( hoh )


Number of models: 1
Number of chains in 1st Model: 1

Chain ID: C
	Number of Residues: 8
	Unique List of Non-A.A.: {'HOH'}


## Extract Metals

In [152]:
from copy import deepcopy
empty_pdb = deepcopy( pdb ) 
print( empty_pdb)
for _ in empty_pdb: del empty_pdb[0]
print( empty_pdb)
new_pdb = empty_pdb
print( new_pdb )
print( id( new_pdb ))
print(id(empty_pdb) )
print(id( pdb ) )


<gemmi.Structure A0152a with 1 model(s)>
<gemmi.Structure A0152a with 0 model(s)>
<gemmi.Structure A0152a with 0 model(s)>
94013157598560
94013157598560
139822113724176


In [180]:
new_model = deepcopy(pdb[0])
print( new_model, new_model.num)
while len(new_model) > 0: del new_model[0]
print( new_model, new_model.num)

<gemmi.Model 1 with 5 chain(s)> 1
<gemmi.Model 1 with 0 chain(s)> 1


In [ ]:
lst = [ 1,2,3, 4, 5,6,7 ]
while len(lst) > 0: del lst[0]
print( lst)


[]


In [161]:
new_chain = deepcopy(pdb[0]["A"])
del new_chain[-1]
empty_resi = deepcopy(new_chain[-1])
for _ in empty_resi: del empty_resi[0] # Only keep residue level
print( empty_resi, len(empty_resi ))

146(GLU) 0


In [204]:
chain = gemmi.Chain("A")
res1, res2, res3 = [gemmi.Residue() for _ in range(3)]
res1.name, res1.seqid = "ALA", gemmi.SeqId(1, "d")
res2.name, res2.seqid = "GLY", gemmi.SeqId(2, "d")
res3.name, res3.seqid = "SER", gemmi.SeqId(3, "d")
for res in [res1, res2, res3]:
    chain.add_residue( res )
    print( chain[-1])
# chain.add_residue( gemmi.Residue( ) )

1d(ALA)
2d(GLY)
3d(SER)


In [213]:
def sele_metal( residue: gemmi.Residue ):
    return [ atom for atom in residue if atom.element.is_metal ]

mini_pdb = createPDB( atomList= [ atom for atom in pdb[0]["A"][0] ])
metal = sele_pdb(mini_pdb, "atom", sele_metal)
print(  len(metal[0]["A"][0]))

mini_pdb = createPDB( atomList= [ atom for atom in pdb[0]["B"][0] ])
metal = sele_pdb(mini_pdb, "atom", sele_metal)
print(  metal[0]["A"][0][0])


get_pdb_stats( metal )

0
<gemmi.Atom ZN at (10.3, -3.9, 5.6)>
Number of models: 1
Number of chains in 1st Model: 1

Chain ID: A
	Number of Residues: 1
	Unique List of Non-A.A.: {'MOL'}


In [12]:

def sele_metal( residue: gemmi.Residue ):
    return [ atom for atom in residue if atom.element.is_metal ]

def sele_HOH( chain: gemmi.Chain ):
    return [ res for res in chain if res.name == "HOH" ]

metal = sele_pdb( pdb, "atom", sele_metal)


# metal= sele_pdb( pdb, "residue", sele_HOH)

get_pdb_stats(metal)

Number of models: 1
Number of chains in 1st Model: 1

Chain ID: B
	Number of Residues: 1
	Unique List of Non-A.A.: {'ZN'}


## Extract Others

In [124]:
def sele_others( chain: gemmi.Chain ):
    return [ res for res in chain if 
            not gemmi.find_tabulated_residue(res.name).is_amino_acid() 
            and res.name not in ["HOH", "LIG" ] ]

others = sele_pdb( pdb, "residue", sele_others)
get_pdb_stats( others )

Number of models: 1
Number of chains in 1st Model: 3

Chain ID: B
	Number of Residues: 1
	Unique List of Non-A.A.: {'ZN'}
Chain ID: D
	Number of Residues: 3
	Unique List of Non-A.A.: {'DMS'}
Chain ID: E
	Number of Residues: 1
	Unique List of Non-A.A.: {'SO4'}


## Extract Binding A.A.

In [3]:
import os 
import sys
sys.path.insert( 0, "../../..")
from pathlib import Path

iePDBPath = """../../../data/ev2a/01-curated/03-refine/01-pdb/04-bound/\
x5240-refine.split.bound-state.pdb"""

iePDBPath = ("../../../data/ev2a/fragalysis/concatDir/aligned_files/"\
        "A0152a/A0152a_apo-desolv.pdb")

iePDBPath = ("../../../data/ev2a/fragalysis/concatDir/aligned_files/"\
        "A0152a/A0152a.pdb")

iePDBPath = Path( iePDBPath)
print( iePDBPath.exists())

True


In [23]:
import gemmi

from xaidar.data.molecModels import sele_res, get_res_CoM

pdb = gemmi.read_pdb( str(iePDBPath))

ligList = sele_res( pdb, {"resname": ["LIG"]})
print( "Number of Ligands id: {}".format( len(ligList) ) )
print( type(ligList[0]))

Number of Ligands id: 1
<class 'gemmi.Residue'>


In [5]:
lig_com = get_res_CoM( ligList[0])

In [7]:
lig_com_pos = gemmi.Position( lig_com[0], lig_com[1], lig_com[2])

In [8]:
pdb[0]["A"][0].get_ca().pos.dist( lig_com_pos )

17.976686309620963

In [ ]:
new_strct = gemmi.Residue( )
new_strct.name = "molec"

print( new_strct )

?(LIG)


In [1]:
from pathlib import Path
import gemmi

rootDataPath = Path( ("../../../data/ev2a/00-test")).resolve()
count = 0
for files in rootDataPath.glob("x0194-all-refine_ground.pdb"):
    model = gemmi.read_pdb(str(files))  
    count += 1

print(count)




1


In [ ]:
model[0][0][0].get_ca().pos

<gemmi.Position(19.238, 6.625, 23.877)>

In [ ]:
model[0].calculate_center_of_mass()

<gemmi.Position(14.5645, 7.2492, 14.869)>

In [ ]:
for i in range( 10):
    if i%3 == 0:
        continue
    print( i )

1
2
4
5
7
8
